# Stand up Auto Loader against a UC Volume, drop a file with a new column, and prove the bronze table evolved without manual intervention

The retail team's hourly product feed has been silently failing for two weeks because the upstream vendor added a `discount_pct` field and the team's brittle JSON-to-Parquet job throws a `cannot resolve column` error on every new file. The on-call gets paged at 7am every Saturday. You have one session to switch the ingest to Auto Loader with schema evolution, prove a new column lands automatically, and write up the configuration so the next vendor change does not page anyone.

The hands-on work:

- Create two UC Volumes (one for source files, one for the Auto Loader `schemaLocation` checkpoint state).
- Upload three baseline product JSON files with a stable schema and run Auto Loader against them with `cloudFiles.schemaEvolutionMode=addNewColumns`.
- Drop a fourth file that adds a `discount_pct` column. Re-run the same query and confirm the bronze table evolved without any `ALTER TABLE` statements.
- Inspect the `schemaLocation` directory and prove the schema file rewrote itself with the new column.

**Time.** Plan for about 60 minutes hands-on. Auto Loader's restart-on-new-column behavior takes one full re-run to land; budget up to 100 minutes for the session window.

**Cost.** Zero dollars. Free Edition Auto Loader runs in directory-listing mode against UC Volumes at no extra charge. Two trigger.availableNow runs of 4 small JSON files each is not how you burn a daily quota.

This lab maps to Databricks DE Associate Domain 2: Data Ingestion and Loading (~20% of exam weight, provisional).

**Runtime note.** This notebook drives Databricks via PAT for SDK and SQL calls AND uses the in-notebook `spark` session for the Auto Loader stream itself. Auto Loader needs PySpark Structured Streaming, which means this notebook must run inside a Databricks workspace notebook attached to serverless all-purpose compute. The setup cell guards against running outside Databricks.

In [ ]:
# NBVAL_SKIP
# Install labrun-checks and the Databricks SDK. Pinned versions per
# LAB-CREATION-BLUEPRINT.md build rules.

!pip install --quiet labrun-checks==0.6.0 databricks-sdk==0.40.0

In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants. Source and checkpoint volumes are separate
# so the `schemaLocation` directory never sits inside the data source path.

import atexit
import getpass
import io
import json
import sys
import time

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.sql import StatementState

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CheckpointResult,
    CleanupResource,
)

LAB_ID = "auto-loader-with-schema-evolution"
LAB_TAG_KEY = "labrun_lab_slug"
LAB_TAG_VALUE = LAB_ID

CATALOG = "workspace"
PARENT_SCHEMA = "default"
LAB_SCHEMA = "labrun_auto_loader"
LAB_SCHEMA_FQN = f"{CATALOG}.{PARENT_SCHEMA}.{LAB_SCHEMA}"

SOURCE_VOLUME = "source_files"
CHECKPOINTS_VOLUME = "checkpoints"
SOURCE_VOLUME_FQN = f"{LAB_SCHEMA_FQN}.{SOURCE_VOLUME}"
CHECKPOINTS_VOLUME_FQN = f"{LAB_SCHEMA_FQN}.{CHECKPOINTS_VOLUME}"

SOURCE_DATE_PREFIX = "2026-05-13"
SOURCE_VOLUME_PATH = f"/Volumes/{CATALOG}/{PARENT_SCHEMA}/{LAB_SCHEMA}/{SOURCE_VOLUME}/{SOURCE_DATE_PREFIX}"
CHECKPOINT_VOLUME_BASE = f"/Volumes/{CATALOG}/{PARENT_SCHEMA}/{LAB_SCHEMA}/{CHECKPOINTS_VOLUME}"
SCHEMA_LOCATION_PATH = f"{CHECKPOINT_VOLUME_BASE}/bronze_products"

BRONZE_TABLE = "bronze_products"
BRONZE_TABLE_FQN = f"{LAB_SCHEMA_FQN}.{BRONZE_TABLE}"

# Deterministic baseline files. Three with the stable schema, one that adds
# discount_pct.
BASELINE_PRODUCTS = [
    {"product_id": "P-001", "product_name": "Stadium Loafer", "unit_price": 119.99},
    {"product_id": "P-002", "product_name": "Field Jacket", "unit_price": 249.50},
    {"product_id": "P-003", "product_name": "Trail Runner", "unit_price": 89.00},
]
NEW_COLUMN_PRODUCT = {
    "product_id": "P-004",
    "product_name": "Court Sneaker",
    "unit_price": 159.99,
    "discount_pct": 0.10,
}

In [ ]:
# NBVAL_SKIP
# Register the session, validate Databricks credentials via the SDK, resolve
# the Starter SQL warehouse, and check that this notebook is running inside
# Databricks (the Auto Loader stream needs the in-notebook spark session).

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
databricks_host = getpass.getpass("Databricks workspace URL: ").strip()
databricks_token = getpass.getpass("Databricks personal access token (PAT): ").strip()

if not databricks_host or not databricks_token:
    print("Workspace URL and PAT are both required. Re-run this cell.")
    raise SystemExit(1)
if not databricks_host.startswith("https://"):
    databricks_host = f"https://{databricks_host}"

w = WorkspaceClient(host=databricks_host, token=databricks_token)

try:
    me = w.current_user.me()
except Exception as exc:
    print("Databricks credentials are missing or expired:")
    print(f"  {exc}")
    raise SystemExit(1)
CALLER_USER = me.user_name or (me.emails[0].value if me.emails else "unknown")
print(f"Credentials validated. Workspace user: {CALLER_USER}")

warehouses = list(w.warehouses.list())
if not warehouses:
    print("No SQL warehouses found. Recreate the Starter warehouse and re-run.")
    raise SystemExit(1)
WAREHOUSE = warehouses[0]
WAREHOUSE_ID = WAREHOUSE.id
print(f"SQL warehouse resolved: {WAREHOUSE.name} ({WAREHOUSE_ID})")

# Auto Loader needs the in-notebook spark session. Outside Databricks, this
# global is undefined and the lab cannot proceed.
try:
    spark  # type: ignore[name-defined]
except NameError:
    print("BLOCKED: this notebook must run inside a Databricks workspace.")
    print("The Auto Loader stream uses spark.readStream which requires the")
    print("Databricks runtime. Import this .ipynb into your Free Edition")
    print("workspace, attach to serverless all-purpose compute, and re-run.")
    raise SystemExit(1)
print("Spark session detected. Auto Loader will run on attached compute.")

DATABRICKS_CREDENTIALS = {
    "host": databricks_host,
    "token": databricks_token,
    "warehouse_id": WAREHOUSE_ID,
}


def run_sql(statement, warehouse_id=None, wait_seconds=180):
    wid = warehouse_id or WAREHOUSE_ID
    resp = w.statement_execution.execute_statement(
        warehouse_id=wid, statement=statement, wait_timeout="50s",
    )
    statement_id = resp.statement_id
    deadline = time.time() + wait_seconds
    while True:
        state = resp.status.state if resp.status else None
        if state == StatementState.SUCCEEDED:
            break
        if state in (StatementState.FAILED, StatementState.CANCELED, StatementState.CLOSED):
            err = resp.status.error.message if (resp.status and resp.status.error) else "no error message"
            raise RuntimeError(f"SQL failed ({state}): {err}\n  Statement: {statement}")
        if time.time() > deadline:
            raise TimeoutError(f"SQL did not finish in {wait_seconds}s. Last state: {state}.")
        time.sleep(2)
        resp = w.statement_execution.get_statement(statement_id)
    columns = []
    if resp.manifest and resp.manifest.schema and resp.manifest.schema.columns:
        columns = [c.name for c in resp.manifest.schema.columns]
    rows = []
    if resp.result and resp.result.data_array:
        rows = list(resp.result.data_array)
    return {"columns": columns, "rows": rows}


register(session_token=session_token, lab_id=LAB_ID, credentials=DATABRICKS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, and orphan scan. Manifest order:
# bronze table first, then volume contents (purge before the volume itself
# can be dropped), then the volumes, then the schema. Auto Loader's
# schemaLocation lives inside the checkpoints volume, so purging the volume
# contents handles the schema-file directory too.


CLEANUP_MANIFEST = [
    CleanupResource(
        type="uc_managed_table",
        id=BRONZE_TABLE_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP TABLE IF EXISTS {BRONZE_TABLE_FQN}\"",
    ),
    CleanupResource(
        type="uc_volume_contents",
        id=CHECKPOINTS_VOLUME_FQN,
        region="databricks",
        cli_delete_command=f"databricks fs rm -r dbfs:{CHECKPOINT_VOLUME_BASE}/",
    ),
    CleanupResource(
        type="uc_volume_contents",
        id=SOURCE_VOLUME_FQN,
        region="databricks",
        cli_delete_command=f"databricks fs rm -r dbfs:/Volumes/{CATALOG}/{PARENT_SCHEMA}/{LAB_SCHEMA}/{SOURCE_VOLUME}/",
    ),
    CleanupResource(
        type="uc_volume",
        id=CHECKPOINTS_VOLUME_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP VOLUME IF EXISTS {CHECKPOINTS_VOLUME_FQN}\"",
    ),
    CleanupResource(
        type="uc_volume",
        id=SOURCE_VOLUME_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP VOLUME IF EXISTS {SOURCE_VOLUME_FQN}\"",
    ),
    CleanupResource(
        type="uc_schema",
        id=LAB_SCHEMA_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP SCHEMA IF EXISTS {LAB_SCHEMA_FQN} CASCADE\"",
    ),
]


class DatabricksCleanupAdapter:
    def delete_resource(self, credentials, resource):
        rtype, rid = resource.type, resource.id
        if rtype == "uc_managed_table":
            run_sql(f"DROP TABLE IF EXISTS {rid}")
        elif rtype == "uc_volume":
            run_sql(f"DROP VOLUME IF EXISTS {rid}")
        elif rtype == "uc_volume_contents":
            volume_path = "/Volumes/" + rid.replace(".", "/") + "/"
            try:
                self._recursive_clear(volume_path)
            except Exception:
                pass
        elif rtype == "uc_schema":
            run_sql(f"DROP SCHEMA IF EXISTS {rid} CASCADE")
        else:
            raise ValueError(f"Unknown resource type {rtype!r}")

    def _recursive_clear(self, path):
        try:
            listing = list(w.files.list_directory_contents(path))
        except Exception:
            return
        for entry in listing:
            if entry.is_directory:
                self._recursive_clear(entry.path)
                try:
                    w.files.delete_directory(entry.path)
                except Exception:
                    pass
            else:
                try:
                    w.files.delete(entry.path)
                except Exception:
                    pass

    def describe_resource(self, credentials, resource):
        rtype, rid = resource.type, resource.id
        if rtype == "uc_managed_table":
            catalog, schema, table = rid.split(".")
            sql = (
                "SELECT 1 FROM system.information_schema.tables "
                f"WHERE table_catalog = '{catalog}' AND table_schema = '{schema}' "
                f"AND table_name = '{table}'"
            )
            return len(run_sql(sql)["rows"]) > 0
        if rtype == "uc_volume":
            catalog, schema, volume = rid.split(".")
            sql = (
                "SELECT 1 FROM system.information_schema.volumes "
                f"WHERE volume_catalog = '{catalog}' AND volume_schema = '{schema}' "
                f"AND volume_name = '{volume}'"
            )
            return len(run_sql(sql)["rows"]) > 0
        if rtype == "uc_volume_contents":
            volume_path = "/Volumes/" + rid.replace(".", "/") + "/"
            try:
                return len(list(w.files.list_directory_contents(volume_path))) > 0
            except Exception:
                return False
        if rtype == "uc_schema":
            catalog, schema = rid.split(".")
            sql = (
                "SELECT 1 FROM system.information_schema.schemata "
                f"WHERE catalog_name = '{catalog}' AND schema_name = '{schema}'"
            )
            return len(run_sql(sql)["rows"]) > 0
        return False

    def tag_scan(self, credentials, lab_slug, region):
        arns = []
        queries = [
            (
                "SELECT catalog_name, schema_name FROM system.information_schema.schema_tags "
                f"WHERE tag_name = '{LAB_TAG_KEY}' AND tag_value = '{lab_slug}'",
                lambda row: f"{row[0]}.{row[1]}",
            ),
            (
                "SELECT catalog_name, schema_name, table_name FROM system.information_schema.table_tags "
                f"WHERE tag_name = '{LAB_TAG_KEY}' AND tag_value = '{lab_slug}'",
                lambda row: f"{row[0]}.{row[1]}.{row[2]}",
            ),
            (
                "SELECT catalog_name, schema_name, volume_name FROM system.information_schema.volume_tags "
                f"WHERE tag_name = '{LAB_TAG_KEY}' AND tag_value = '{lab_slug}'",
                lambda row: f"{row[0]}.{row[1]}.{row[2]}",
            ),
        ]
        for sql, fmt in queries:
            try:
                result = run_sql(sql)
            except Exception:
                continue
            for row in result["rows"]:
                arns.append(fmt(row))
        return arns


CLEANUP_ADAPTER = DatabricksCleanupAdapter()


def _atexit_cleanup():
    try:
        run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


orphans = CLEANUP_ADAPTER.tag_scan(DATABRICKS_CREDENTIALS, LAB_TAG_VALUE, "databricks")
if orphans:
    print(f"BLOCKED: existing UC objects tagged {LAB_TAG_KEY}={LAB_TAG_VALUE} were found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("Run the cleanup cell at the bottom first, then re-run setup.")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")

## Task 1: Stand up the schema, both volumes, and the three baseline files

Two volumes, not one. The source volume holds the JSON the team drops; the checkpoints volume holds the Auto Loader `schemaLocation` (the schema files that record what columns Auto Loader has seen). Co-locating them inside the data source is a common trap; the validator checks the layout.

Build it in this order:

1. `CREATE SCHEMA IF NOT EXISTS workspace.default.labrun_auto_loader` and tag it.
2. `CREATE VOLUME IF NOT EXISTS workspace.default.labrun_auto_loader.source_files` (a managed volume) and tag it.
3. `CREATE VOLUME IF NOT EXISTS workspace.default.labrun_auto_loader.checkpoints` (a managed volume) and tag it.
4. Upload three baseline product JSON files under `/Volumes/.../source_files/2026-05-13/product-001.json`, `-002.json`, `-003.json`. Each file holds a single JSON object with keys `product_id`, `product_name`, `unit_price`. Use `w.files.upload(file_path=..., contents=io.BytesIO(...), overwrite=True)` from the SDK.

The validator confirms the volumes are tagged and the three baseline files exist with the expected schema.

In [ ]:
# NBVAL_SKIP
# Task 1: Schema, volumes, baseline files. Tag every UC object so the
# orphan scan can find them.

# YOUR CODE: Run CREATE SCHEMA IF NOT EXISTS for LAB_SCHEMA_FQN via run_sql().

# YOUR CODE: Run ALTER SCHEMA LAB_SCHEMA_FQN SET TAGS
# ('labrun_lab_slug' = LAB_TAG_VALUE) via run_sql().

# YOUR CODE: Run CREATE VOLUME IF NOT EXISTS for SOURCE_VOLUME_FQN
# via run_sql(). Use the managed-volume syntax (no LOCATION clause).

# YOUR CODE: Run ALTER VOLUME SOURCE_VOLUME_FQN SET TAGS
# ('labrun_lab_slug' = LAB_TAG_VALUE) via run_sql().

# YOUR CODE: Run CREATE VOLUME IF NOT EXISTS for CHECKPOINTS_VOLUME_FQN
# via run_sql() (managed volume).

# YOUR CODE: Run ALTER VOLUME CHECKPOINTS_VOLUME_FQN SET TAGS
# ('labrun_lab_slug' = LAB_TAG_VALUE) via run_sql().

# Upload the three baseline files. Each file holds a single JSON object.
for idx, product in enumerate(BASELINE_PRODUCTS, start=1):
    file_name = f"product-{idx:03d}.json"
    file_path = f"{SOURCE_VOLUME_PATH}/{file_name}"
    payload = json.dumps(product).encode("utf-8")
    # YOUR CODE: Upload the file via
    # w.files.upload(file_path=file_path, contents=io.BytesIO(payload),
    # overwrite=True).
    print(f"Uploaded {file_path}")

print()
print(f"Baseline staged under {SOURCE_VOLUME_PATH}/")

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: Schema, source volume, checkpoints volume all exist and are
# tagged; three product-*.json baseline files exist under the source volume.


def checkpoint_1(session):
    try:
        schema_sql = (
            "SELECT 1 FROM system.information_schema.schemata "
            f"WHERE catalog_name = '{CATALOG}' AND schema_name = '{LAB_SCHEMA}'"
        )
        if not run_sql(schema_sql)["rows"]:
            return CheckpointResult(
                status="fail",
                error_reason=f"Schema {LAB_SCHEMA_FQN} not found.",
            )

        vol_sql = (
            "SELECT volume_name FROM system.information_schema.volumes "
            f"WHERE volume_catalog = '{CATALOG}' AND volume_schema = '{LAB_SCHEMA}'"
        )
        volumes_present = {row[0] for row in run_sql(vol_sql)["rows"]}
        missing = {SOURCE_VOLUME, CHECKPOINTS_VOLUME} - volumes_present
        if missing:
            return CheckpointResult(
                status="fail",
                error_reason=f"Missing volume(s) in {LAB_SCHEMA_FQN}: {sorted(missing)}.",
            )

        try:
            listing = list(w.files.list_directory_contents(SOURCE_VOLUME_PATH + "/"))
        except Exception as exc:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Could not list source volume path {SOURCE_VOLUME_PATH}/: {exc}. "
                    f"Confirm the three baseline files uploaded to the dated subdirectory."
                ),
            )
        json_files = [e for e in listing if e.path.endswith(".json")]
        if len(json_files) < 3:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Source path {SOURCE_VOLUME_PATH}/ has {len(json_files)} *.json "
                    f"files; expected at least 3 baseline files."
                ),
            )

        sample = json_files[0].path
        try:
            response = w.files.download(sample)
            body = response.contents.read()
            parsed = json.loads(body)
        except Exception as exc:
            return CheckpointResult(
                status="fail",
                error_reason=f"Could not parse {sample} as JSON: {exc}.",
            )
        required_keys = {"product_id", "product_name", "unit_price"}
        if not required_keys.issubset(set(parsed.keys())):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Baseline file {sample} is missing keys; found {sorted(parsed.keys())}, "
                    f"expected at least {sorted(required_keys)}."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint fails on whichever piece is missing: schema, volume, or files. Read the message before peeking. If the files are missing, confirm you uploaded under the `2026-05-13/` subdirectory, not directly under the volume root.

</details>

<details><summary>Hint 2 (direction)</summary>

Six SQL statements, three file uploads. The SQL: CREATE SCHEMA, ALTER SCHEMA SET TAGS, CREATE VOLUME (source), ALTER VOLUME SET TAGS (source), CREATE VOLUME (checkpoints), ALTER VOLUME SET TAGS (checkpoints). The file uploads use `w.files.upload(file_path=..., contents=io.BytesIO(json.dumps(product).encode("utf-8")), overwrite=True)`. The file_path is `f"{SOURCE_VOLUME_PATH}/{file_name}"`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 1:

```python
run_sql(f"CREATE SCHEMA IF NOT EXISTS {LAB_SCHEMA_FQN}")
run_sql(f"ALTER SCHEMA {LAB_SCHEMA_FQN} SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')")

run_sql(f"CREATE VOLUME IF NOT EXISTS {SOURCE_VOLUME_FQN}")
run_sql(f"ALTER VOLUME {SOURCE_VOLUME_FQN} SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')")

run_sql(f"CREATE VOLUME IF NOT EXISTS {CHECKPOINTS_VOLUME_FQN}")
run_sql(f"ALTER VOLUME {CHECKPOINTS_VOLUME_FQN} SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')")

for idx, product in enumerate(BASELINE_PRODUCTS, start=1):
    file_name = f"product-{idx:03d}.json"
    file_path = f"{SOURCE_VOLUME_PATH}/{file_name}"
    payload = json.dumps(product).encode("utf-8")
    w.files.upload(file_path=file_path, contents=io.BytesIO(payload), overwrite=True)
```

</details>

**Wallet check.** Still at $0.00. Two volumes are metadata records; three tiny JSON files are a few hundred bytes. Free Edition has not noticed.

## Task 2: Configure Auto Loader and run the initial ingest

The Auto Loader pattern is one reader plus one writer:

```python
(spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", SCHEMA_LOCATION_PATH)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("cloudFiles.inferColumnTypes", "true")
    .load(SOURCE_VOLUME_PATH + "/")
    .writeStream
    .option("checkpointLocation", SCHEMA_LOCATION_PATH)
    .trigger(availableNow=True)
    .toTable(BRONZE_TABLE_FQN))
```

Key options:

- `cloudFiles.format` is `json`.
- `cloudFiles.schemaLocation` is a per-table directory under the checkpoints volume. NEVER share this between streams; doing so silently corrupts both schemas (the reflection MCQ at the bottom covers this).
- `cloudFiles.schemaEvolutionMode=addNewColumns` is the configuration the exam tests. The reflection cell covers what it actually does on first encounter with a new column.
- `cloudFiles.inferColumnTypes=true` lets Auto Loader pick DOUBLE for `unit_price` instead of STRING.
- `trigger(availableNow=True)` is the only batch-style trigger Free Edition exposes.

After the stream completes, `SELECT count(*)` against `bronze_products` should return 3. If you see 0, the stream did not run or wrote to a different path.

In [ ]:
# NBVAL_SKIP
# Task 2: Build and run the Auto Loader stream against the three baseline
# files with trigger(availableNow=True).

# YOUR CODE: Build the Auto Loader readStream pipeline and call .toTable()
# on it with trigger(availableNow=True). The full configuration is in the
# task markdown above; assign the streaming query result to `query` and
# call query.awaitTermination() so the cell blocks until the trigger
# completes.

print("Auto Loader trigger.availableNow run complete.")

count_rows = run_sql(f"SELECT count(*) FROM {BRONZE_TABLE_FQN}")["rows"]
print(f"{BRONZE_TABLE_FQN} row count: {int(count_rows[0][0])}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: Bronze table exists with the expected columns and three rows.


def checkpoint_2(session):
    try:
        table_sql = (
            "SELECT table_type, data_source_format "
            "FROM system.information_schema.tables "
            f"WHERE table_catalog = '{CATALOG}' AND table_schema = '{LAB_SCHEMA}' "
            f"AND table_name = '{BRONZE_TABLE}'"
        )
        rows = run_sql(table_sql)["rows"]
        if not rows:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Table {BRONZE_TABLE_FQN} not found. Did the Auto Loader "
                    f"writer's toTable() target the right name?"
                ),
            )
        table_type, fmt = rows[0]
        if table_type != "MANAGED" or (fmt or "").upper() != "DELTA":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Table {BRONZE_TABLE_FQN} is {table_type}/{fmt}; expected MANAGED/DELTA."
                ),
            )

        col_rows = run_sql(
            "SELECT column_name FROM system.information_schema.columns "
            f"WHERE table_catalog = '{CATALOG}' AND table_schema = '{LAB_SCHEMA}' "
            f"AND table_name = '{BRONZE_TABLE}'"
        )["rows"]
        cols = {row[0] for row in col_rows}
        required = {"product_id", "product_name", "unit_price"}
        missing = required - cols
        if missing:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"{BRONZE_TABLE_FQN} is missing columns {sorted(missing)}. "
                    f"Auto Loader should have inferred them from the JSON keys."
                ),
            )

        count_rows = run_sql(f"SELECT count(*) FROM {BRONZE_TABLE_FQN}")["rows"]
        row_count = int(count_rows[0][0]) if count_rows else 0
        if row_count != 3:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"{BRONZE_TABLE_FQN} has {row_count} rows; expected 3. "
                    f"Either the stream did not run or the schemaLocation was "
                    f"already populated from a previous attempt. Run the cleanup "
                    f"cell and start over if you suspect the latter."
                ),
            )

        try:
            schema_listing = list(w.files.list_directory_contents(SCHEMA_LOCATION_PATH))
        except Exception:
            schema_listing = []
        if not schema_listing:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Auto Loader did not write any state under {SCHEMA_LOCATION_PATH}. "
                    f"Confirm cloudFiles.schemaLocation and checkpointLocation point "
                    f"at this path."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint message names the failure shape: missing table (the writer wrote elsewhere), wrong table type, wrong column set, or wrong row count. If everything except the row count is right, your stream did not run; check that you called `.awaitTermination()` on the streaming query.

</details>

<details><summary>Hint 2 (direction)</summary>

The reader is `spark.readStream.format("cloudFiles").option(...)...load(SOURCE_VOLUME_PATH + "/")`. The writer is `.writeStream.option("checkpointLocation", SCHEMA_LOCATION_PATH).trigger(availableNow=True).toTable(BRONZE_TABLE_FQN)`. The `cloudFiles.schemaLocation` and the `checkpointLocation` can be the same path for a single-table stream. Wrap the whole thing in parentheses or chain with backslashes. Assign the return value to a variable and call `.awaitTermination()` so the cell blocks.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 2:

```python
query = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", SCHEMA_LOCATION_PATH)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("cloudFiles.inferColumnTypes", "true")
    .load(SOURCE_VOLUME_PATH + "/")
    .writeStream.option("checkpointLocation", SCHEMA_LOCATION_PATH)
    .trigger(availableNow=True)
    .toTable(BRONZE_TABLE_FQN)
)
query.awaitTermination()
```

</details>

**Wallet check.** Still at $0.00. trigger.availableNow ran one pass against three small JSON files. The serverless attachment is the only thing burning, and serverless on Free Edition is part of the daily quota, not a billable line item.

## Task 3: Drop the new-column file and re-run the same query

This is the lesson. Upload a fourth file with a `discount_pct` field that none of the baseline files had. Re-run the EXACT same Auto Loader reader and writer (no `ALTER TABLE`, no schema hints). `schemaEvolutionMode=addNewColumns` does this:

1. First run after the new file lands: Auto Loader throws `UnknownFieldException`, writes a new schema file under `schemaLocation/_schemas/`, and stops the stream.
2. Second run (same code): Auto Loader reads the updated schema, ingests the new file, and the bronze table now has the `discount_pct` column populated for the new row and NULL for the older rows.

In `trigger(availableNow=True)` mode the "restart" is a re-run of the cell. You may see an exception on the first run, then success on the second. The validator allows either behavior on a single cell run: it just checks the end state (4 rows, `discount_pct` populated correctly).

In [ ]:
# NBVAL_SKIP
# Task 3: Upload the fourth file (adds discount_pct), then re-run the Auto
# Loader stream with the same configuration. Run the cell once; if you get
# an UnknownFieldException, re-run it.

new_file_path = f"{SOURCE_VOLUME_PATH}/product-004.json"
payload = json.dumps(NEW_COLUMN_PRODUCT).encode("utf-8")
# YOUR CODE: Upload the fourth file with the same w.files.upload() call as
# Task 1, using file_path=new_file_path, contents=io.BytesIO(payload),
# overwrite=True.
print(f"Uploaded {new_file_path}")

# Re-run the Auto Loader stream. addNewColumns mode either lands the new
# column immediately or restarts once on UnknownFieldException; in
# trigger(availableNow=True) the restart is a manual re-run of this cell.
attempts = 0
max_attempts = 3
while attempts < max_attempts:
    attempts += 1
    try:
        # YOUR CODE: Build the SAME Auto Loader stream from Task 2 (same
        # options, same schemaLocation, same toTable target) and call
        # awaitTermination() on it. Assign to `query` and break on success.
        break
    except Exception as exc:
        message = repr(exc)
        if "UnknownFieldException" in message or "addNewColumns" in message:
            print(f"Attempt {attempts}: Auto Loader saw a new column, restarting the stream...")
            time.sleep(2)
            continue
        raise

count_rows = run_sql(f"SELECT count(*) FROM {BRONZE_TABLE_FQN}")["rows"]
print(f"{BRONZE_TABLE_FQN} row count: {int(count_rows[0][0])}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: Bronze table has 4 rows; column discount_pct is present;
# the row for product-004.json has discount_pct = 0.10; the original three
# rows have discount_pct IS NULL.


def checkpoint_3(session):
    try:
        count_rows = run_sql(f"SELECT count(*) FROM {BRONZE_TABLE_FQN}")["rows"]
        row_count = int(count_rows[0][0]) if count_rows else 0
        if row_count != 4:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"{BRONZE_TABLE_FQN} has {row_count} rows; expected 4 after "
                    f"product-004.json ingest. Re-run Task 3's stream cell once "
                    f"more if Auto Loader threw UnknownFieldException on the "
                    f"first attempt."
                ),
            )

        col_rows = run_sql(
            "SELECT column_name FROM system.information_schema.columns "
            f"WHERE table_catalog = '{CATALOG}' AND table_schema = '{LAB_SCHEMA}' "
            f"AND table_name = '{BRONZE_TABLE}'"
        )["rows"]
        cols = {row[0] for row in col_rows}
        if "discount_pct" not in cols:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"{BRONZE_TABLE_FQN} does not have a discount_pct column. "
                    f"Auto Loader's addNewColumns mode should have added it. "
                    f"Confirm cloudFiles.schemaEvolutionMode is set to 'addNewColumns' "
                    f"and that you re-ran the stream after uploading product-004.json."
                ),
            )

        new_row = run_sql(
            f"SELECT discount_pct FROM {BRONZE_TABLE_FQN} "
            f"WHERE product_id = 'P-004'"
        )["rows"]
        if not new_row:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"No row for product_id='P-004' found in {BRONZE_TABLE_FQN}. "
                    f"Confirm product-004.json was uploaded with the expected payload."
                ),
            )
        discount = new_row[0][0]
        if discount is None or abs(float(discount) - 0.10) > 1e-6:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"P-004 has discount_pct={discount!r}; expected 0.10. "
                    f"Confirm the uploaded JSON payload for product-004.json."
                ),
            )

        null_rows = run_sql(
            f"SELECT count(*) FROM {BRONZE_TABLE_FQN} "
            f"WHERE discount_pct IS NULL AND product_id IN ('P-001', 'P-002', 'P-003')"
        )["rows"]
        null_count = int(null_rows[0][0]) if null_rows else 0
        if null_count != 3:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Expected the three original rows (P-001, P-002, P-003) to "
                    f"have discount_pct IS NULL; found {null_count} matching. "
                    f"If older rows somehow have a value, the column was hand-populated."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint message names which piece broke: row count (the new file did not get picked up), missing column (schemaEvolutionMode was not set to addNewColumns), wrong value on P-004 (the file payload was wrong), or wrong nulls on older rows (somebody hand-filled the column). Read the message before peeking.

</details>

<details><summary>Hint 2 (direction)</summary>

Two things to do. (1) Upload `product-004.json` via `w.files.upload(file_path=new_file_path, contents=io.BytesIO(payload), overwrite=True)`. (2) Re-run the exact same Auto Loader stream from Task 2. If you get `UnknownFieldException`, run the cell again; that exception is Auto Loader's signal to evolve the schema, and the next run picks up the new column.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 3:

```python
new_file_path = f"{SOURCE_VOLUME_PATH}/product-004.json"
payload = json.dumps(NEW_COLUMN_PRODUCT).encode("utf-8")
w.files.upload(file_path=new_file_path, contents=io.BytesIO(payload), overwrite=True)

# Re-run the stream. If addNewColumns throws on the first attempt, re-run.
for _ in range(3):
    try:
        query = (
            spark.readStream.format("cloudFiles")
            .option("cloudFiles.format", "json")
            .option("cloudFiles.schemaLocation", SCHEMA_LOCATION_PATH)
            .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
            .option("cloudFiles.inferColumnTypes", "true")
            .load(SOURCE_VOLUME_PATH + "/")
            .writeStream.option("checkpointLocation", SCHEMA_LOCATION_PATH)
            .trigger(availableNow=True)
            .toTable(BRONZE_TABLE_FQN)
        )
        query.awaitTermination()
        break
    except Exception as exc:
        if "UnknownFieldException" not in repr(exc) and "addNewColumns" not in repr(exc):
            raise
```

</details>

**Wallet check.** Still at $0.00. One more JSON upload (~120 bytes) and one more trigger.availableNow pass. The Auto Loader schema file rewrite touched a single JSON in the `_schemas/` directory.

## Task 4: Confirm the schemaLocation is unique and the schema file evolved

The validator does this for you in Checkpoint 4; the task is the manual inspection that builds the mental model. List the contents of `schemaLocation` and confirm:

- A `_schemas/` subdirectory exists.
- It contains at least two JSON schema files (one for the baseline three-column schema, one for the addNewColumns evolution).
- The latest schema file lists `discount_pct` as a top-level field.
- The `schemaLocation` path is per-table (`/Volumes/.../checkpoints/bronze_products/`), not shared with any other Auto Loader stream.

The reflection MCQ at the bottom of the notebook tests what happens when two streams share a `schemaLocation`. Read it carefully.

In [ ]:
# NBVAL_SKIP
# Task 4: Walk the schemaLocation and confirm the evolved schema is on disk.

print(f"Listing {SCHEMA_LOCATION_PATH}:")
try:
    top = list(w.files.list_directory_contents(SCHEMA_LOCATION_PATH))
    for entry in top:
        print(f"  {'dir ' if entry.is_directory else 'file'}  {entry.path}")
except Exception as exc:
    print(f"  (could not list: {exc})")

schemas_path = f"{SCHEMA_LOCATION_PATH}/_schemas"
print(f"\nListing {schemas_path}:")
schema_files = []
try:
    for entry in w.files.list_directory_contents(schemas_path):
        if not entry.is_directory:
            schema_files.append(entry.path)
        print(f"  file  {entry.path}")
except Exception as exc:
    print(f"  (could not list: {exc})")

if schema_files:
    schema_files.sort()
    latest = schema_files[-1]
    print(f"\nLatest schema file: {latest}")
    try:
        response = w.files.download(latest)
        body = response.contents.read().decode("utf-8")
        print(body)
    except Exception as exc:
        print(f"(could not read schema file: {exc})")

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: schemaLocation has a _schemas/ subdirectory with at least
# two schema files, the latest mentions discount_pct, and the lab's
# checkpoints volume has only one stream's schemaLocation (no sibling
# directories that would indicate a shared path across tables).


def checkpoint_4(session):
    try:
        schemas_path = f"{SCHEMA_LOCATION_PATH}/_schemas"
        try:
            schema_entries = list(w.files.list_directory_contents(schemas_path))
        except Exception as exc:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Could not list Auto Loader _schemas directory at {schemas_path}: {exc}. "
                    f"Confirm cloudFiles.schemaLocation is set to {SCHEMA_LOCATION_PATH}."
                ),
            )
        schema_files = [e.path for e in schema_entries if not e.is_directory]
        if len(schema_files) < 2:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Auto Loader _schemas/ has {len(schema_files)} schema file(s); "
                    f"expected at least 2 (one for the baseline, one for the evolution). "
                    f"Re-run Task 3 so the schema file rewrites."
                ),
            )

        schema_files.sort()
        latest = schema_files[-1]
        try:
            response = w.files.download(latest)
            body = response.contents.read().decode("utf-8")
        except Exception as exc:
            return CheckpointResult(
                status="error",
                error_reason=f"Could not read {latest}: {exc}",
            )
        if "discount_pct" not in body:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Latest schema file {latest} does not mention 'discount_pct'. "
                    f"The schema evolution did not land; re-run Task 3."
                ),
            )

        try:
            checkpoint_root = list(w.files.list_directory_contents(CHECKPOINT_VOLUME_BASE + "/"))
        except Exception:
            checkpoint_root = []
        peer_dirs = [
            e.path for e in checkpoint_root
            if e.is_directory and e.path.rstrip("/") != SCHEMA_LOCATION_PATH.rstrip("/")
        ]
        peer_schema_dirs = []
        for peer in peer_dirs:
            try:
                contents = list(w.files.list_directory_contents(peer))
            except Exception:
                continue
            for sub in contents:
                if sub.is_directory and sub.path.rstrip("/").endswith("_schemas"):
                    peer_schema_dirs.append(peer)
                    break
        if peer_schema_dirs:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Found additional schemaLocation directories under the checkpoints "
                    f"volume: {peer_schema_dirs}. Each stream needs its own per-table "
                    f"schemaLocation; sharing a path silently corrupts both schemas."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

Most failures on this checkpoint trace back to Task 3 not actually running the second time. If `_schemas/` has only one file, the addNewColumns evolution did not land. Re-run Task 3's stream cell and watch for the row-count change.

</details>

<details><summary>Hint 2 (direction)</summary>

No new code to write for this task. The checkpoint inspects what Auto Loader already wrote. If it complains about peer `schemaLocation` directories, something earlier in this session pointed a second stream at the same checkpoints volume; the cleanest fix is to run the cleanup cell and re-do the lab.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Nothing to write. If the checkpoint still fails after re-running Task 3, run the cleanup cell at the bottom of the notebook and restart from Task 1; the checkpoints volume was likely populated by an earlier attempt with a different schemaLocation path.

</details>

**Wallet check.** Still at $0.00. Listing volume contents and downloading a single tiny schema JSON costs nothing. The session has spent a sliver of the daily compute quota and zero dollars.

## Cleanup

Time to tear it all down. The cell below drops the bronze table, purges both volumes (the schemaLocation directory disappears with the checkpoints volume contents), drops both volumes, then drops the schema with `CASCADE`. Re-running is safe.

In [ ]:
# NBVAL_SKIP
# Cleanup. Tear down every resource in CLEANUP_MANIFEST in reverse-creation
# order using the inline Databricks adapter.

result = run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids = set()
for warning_msg in result.warnings:
    for res in CLEANUP_MANIFEST:
        if res.id in warning_msg:
            failed_ids.add(res.id)
            break

critical_destroyed = 0
standard_destroyed = len(CLEANUP_MANIFEST) - len(failed_ids)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your Databricks workspace may still hold lab objects. Resolve before closing this notebook.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)

**Session total: $0.00.** Two `trigger(availableNow=True)` runs, four JSON files, and a schema evolution that took zero `ALTER TABLE` statements to land. Free Edition swallowed every byte. The expensive thing this session would have replaced is the on-call's 7am Saturday pages.

## Reflection

These are not graded. They are for you.

1. Walk through what Auto Loader does the first time it sees a new column in a JSON file with `cloudFiles.schemaEvolutionMode=addNewColumns`. Name every step: the parse, the `UnknownFieldException`, the stream restart, the schema file rewrite under `schemaLocation`, the bronze-table ALTER. Why is the restart-once behavior preferable to silently widening the schema?

2. Your team is debating Auto Loader vs `COPY INTO` for two distinct sources: a high-frequency clickstream that lands JSON every minute and a daily batch CSV that lands once at 3am. Sketch the decision factors you would weigh (idempotency, schema evolution, latency, operational overhead) and which source goes which way.

## Exam-Style Practice

**Question 1 (Domain 2, Auto Loader schema evolution):**

A data engineer configures Auto Loader to ingest JSON files from a UC Volume into a bronze Delta table. The reader uses `cloudFiles.schemaEvolutionMode=addNewColumns`. A new file lands with a previously unseen column `customer_segment`. The engineer expects the column to be added to the bronze table. What happens on the first run after the new file lands?

A. The bronze table is silently widened to include `customer_segment`; the new row has the value and older rows show NULL. No exception is raised.

B. Auto Loader raises `UnknownFieldException`, marks the schema file with the new column, and stops the stream; the engineer re-runs the same query and the second run loads the new file with the widened schema.

C. The bronze table rejects the new file with `SchemaEnforcementException`; the engineer must run `ALTER TABLE ADD COLUMN customer_segment STRING` and re-run.

D. Auto Loader silently drops the new column from the JSON and writes the row with only the previously-known columns; the `_rescued_data` column captures the dropped value.

<details><summary>Show answer</summary>

**Correct: B.**

- A would be the behavior of `schemaEvolutionMode=rescue` or `schemaEvolutionMode=none` plus a `_rescued_data` column. `addNewColumns` does not silently widen; it restarts.
- B is correct: `addNewColumns` is the documented behavior. The first run after a new column appears throws `UnknownFieldException` and updates the schema file under `cloudFiles.schemaLocation`. The stream is then restartable; the second run picks up the file with the evolved schema. In `trigger(availableNow=True)` mode the engineer simply re-runs the cell.
- C is wrong: `SchemaEnforcementException` is not the relevant exception, and `ALTER TABLE ADD COLUMN` is not required because the whole point of `addNewColumns` is automation.
- D describes `schemaEvolutionMode=rescue`, not `addNewColumns`.

</details>

**Question 2 (Domain 2, schemaLocation isolation):**

A platform engineer configures two Auto Loader streams writing to two different bronze tables (`bronze_orders` and `bronze_customers`) from two different folders in the same UC Volume. To save typing, the engineer sets both streams' `cloudFiles.schemaLocation` to the same path. After running both for a week, the engineer finds bronze_orders has columns that should only exist in bronze_customers, and vice versa. What is the cause?

A. Auto Loader inferred the schema from the wrong folder for both streams; setting `cloudFiles.inferColumnTypes=true` would have prevented it.

B. `cloudFiles.schemaLocation` is the source of truth for the stream's schema and must be unique per target table; sharing the path caused the two streams to overwrite each other's schema files and merge each other's inferred columns.

C. UC Volumes do not support concurrent Auto Loader reads; one stream silently consumed files intended for the other.

D. The folders should have been in two different volumes; sharing a volume across streams is unsupported.

<details><summary>Show answer</summary>

**Correct: B.**

- A is a real Auto Loader option but does not explain cross-stream column contamination; `inferColumnTypes` controls inference per stream, not isolation between streams.
- B is correct: `cloudFiles.schemaLocation` holds the schema files for the stream. Two streams pointed at the same location overwrite each other's `_schemas/*.json` files as new files arrive. Each stream then reads the merged schema and writes the merged columns into its target table.
- C is wrong: UC Volumes fully support concurrent Auto Loader reads on different folders or different prefixes.
- D is wrong: a single volume hosting multiple stream sources is supported; the isolation must happen at the schemaLocation level, not the volume level.

</details>